In [2]:
from datasets import load_dataset
import torch
from transformers import DetrImageProcessor
from tqdm import tqdm


In [3]:
ds = load_dataset("detection-datasets/coco", cache_dir="../data/")


Resolving data files:   0%|          | 0/40 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/20 [00:00<?, ?it/s]

In [4]:

model_name = "facebook/detr-resnet-50"
image_processor = DetrImageProcessor.from_pretrained(model_name)

In [5]:
train_coco = ds["train"]
val_coco = ds["val"]

In [6]:

def _ensure_3ch(img: torch.Tensor) -> torch.Tensor:
    # img: (C,H,W), uint8
    if img.ndim != 3:
        raise ValueError(f"Expected CHW image, got shape {tuple(img.shape)}")

    c, h, w = img.shape
    if c == 3:
        return img
    if c == 1:
        return img.repeat(3, 1, 1)
    if c == 4:
        return img[:3]  # drop alpha
    raise ValueError(f"Unsupported channel count: {c} (shape {tuple(img.shape)})")


def collate_fn(batch):
    # images from dataset (CHW uint8)
    #images = [item["image"] for item in batch]
    images = [_ensure_3ch(item["image"]) for item in batch]

    # build COCO-style annotations per image
    targets = []
    for item in batch:
        boxes_xyxy = item["objects"]["bbox"]          # (N, 4) xyxy
        class_ids  = item["objects"]["category"]      # (N,)
        areas      = item["objects"].get("area", None)
        iscrowd    = item["objects"].get("iscrowd", None)

        # xyxy -> xywh
        x1, y1, x2, y2 = boxes_xyxy.unbind(dim=1)
        boxes_xywh = torch.stack([x1, y1, x2 - x1, y2 - y1], dim=1)

        annotations = []
        for i in range(boxes_xywh.shape[0]):
            ann = {
                "bbox": boxes_xywh[i].tolist(),           # COCO expects python lists
                "category_id": int(class_ids[i].item()),
            }
            if areas is not None:
                ann["area"] = float(areas[i].item())
            if iscrowd is not None:
                ann["iscrowd"] = int(iscrowd[i].item())
            else:
                ann["iscrowd"] = 0
            annotations.append(ann)

        targets.append(
            {"image_id": int(item["image_id"].item()), "annotations": annotations}
        )

    encoded = image_processor(images=images, annotations=targets, return_tensors="pt")

    return {
        "pixel_values": encoded["pixel_values"],     # (B, 3, Hmax, Wmax)
        "pixel_mask": encoded["pixel_mask"],        # (B, Hmax, Wmax)
        "labels": encoded["labels"],          # list[dict] length B
    }

In [7]:
torch_train_ds = train_coco.with_format("torch")
torch_val_ds = val_coco.with_format("torch")

In [8]:
train_loader = torch.utils.data.DataLoader(torch_train_ds, batch_size=2, shuffle=True, collate_fn=collate_fn)
val_loader = torch.utils.data.DataLoader(torch_val_ds, batch_size=2, shuffle=False, collate_fn=collate_fn)

In [9]:

for batch in tqdm(train_loader, desc="Training"):
    pixel_values = batch["pixel_values"]  # (B, 3, Hmax, Wmax)
    pixel_mask = batch["pixel_mask"]      # (B, Hmax, Wmax)
    labels = batch["labels"]              # list[dict] length B

    # Here you would typically pass pixel_values and labels to your model and compute loss
    # For demonstration, we'll just print the shapes
    print(f"Pixel values shape: {pixel_values.shape}")
    print(f"Pixel mask shape: {pixel_mask.shape}")
    print(f"Labels: {labels}")
    break  # Remove this break to process the entire dataset

Training:   0%|          | 0/58633 [00:00<?, ?it/s]

Pixel values shape: torch.Size([2, 3, 800, 1066])
Pixel mask shape: torch.Size([2, 800, 1066])
Labels: [{'size': tensor([ 800, 1066]), 'image_id': tensor([98616]), 'class_labels': tensor([20, 32,  0]), 'boxes': tensor([[0.4678, 0.5784, 0.5654, 0.8206],
        [0.5954, 0.8352, 0.0427, 0.0543],
        [0.4848, 0.8804, 0.1377, 0.2382]]), 'area': tensor([200409.5156,   1467.7549,  17020.9648]), 'iscrowd': tensor([0, 0, 0]), 'orig_size': tensor([640, 640])}, {'size': tensor([ 800, 1066]), 'image_id': tensor([93581]), 'class_labels': tensor([50, 50, 50]), 'boxes': tensor([[0.7374, 0.6423, 0.1736, 0.2696],
        [0.8902, 0.4311, 0.2197, 0.5483],
        [0.3902, 0.4264, 0.7803, 0.8517]]), 'area': tensor([ 30198.0508,  55153.2852, 140538.0625]), 'iscrowd': tensor([0, 0, 0]), 'orig_size': tensor([480, 640])}]


In [10]:

def test_batch_basic(batch):
    assert set(batch.keys()) == {"pixel_values", "pixel_mask", "labels"}

    images = batch["pixel_values"]
    masks = batch["pixel_mask"]
    targets = batch["labels"]

    assert isinstance(images, torch.Tensor)
    assert isinstance(masks, torch.Tensor)
    assert isinstance(targets, list)

    B, C, H, W = images.shape
    assert C == 3
    assert masks.shape == (B, H, W)
    assert len(targets) == B

    assert images.dtype in (torch.float16, torch.float32, torch.bfloat16)
    assert masks.dtype in (torch.bool, torch.uint8, torch.int64, torch.int32)

# run
batch = next(iter(train_loader))
test_batch_basic(batch)
print("✅ basic batch checks passed")

✅ basic batch checks passed


In [11]:
def test_padding_mask_consistency(batch, tol_frac_nonzero=0.02):
    images = batch["pixel_values"]
    masks = batch["pixel_mask"].bool()

    # padded positions: mask == False
    padded = ~masks  # (B,H,W)

    # If nothing padded (rare), pass
    if padded.sum().item() == 0:
        return

    # Check fraction of "significantly nonzero" pixels in padded region
    # images: (B,3,H,W) -> (B,H,W,3)
    img_hw3 = images.permute(0, 2, 3, 1)
    padded_pixels = img_hw3[padded]  # (N,3)

    frac_nonzero = (padded_pixels.abs().mean(dim=1) > 1e-3).float().mean().item()
    assert frac_nonzero < tol_frac_nonzero, f"Too many nonzero padded pixels: {frac_nonzero:.3f}"

# run
test_padding_mask_consistency(batch)
print("✅ padding/mask consistency passed")

✅ padding/mask consistency passed
